# Lab type: debug
# Course: ML203 — Unsupervised Learning & Clustering
# Lesson: DBSCAN
# Task: Find and fix the 3 bugs in the DBSCAN pipeline. After fixing each bug, write a one-sentence explanation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

## Step 1: Generate Non-Spherical Data

In [ ]:
np.random.seed(42)

# Create two crescent-shaped clusters (k-means would fail here)
theta = np.linspace(0, 2*np.pi, 50)
cluster_1 = np.column_stack([5*np.cos(theta) + np.random.normal(0, 0.3, 50),
                              5*np.sin(theta) + np.random.normal(0, 0.3, 50)])

cluster_2 = np.column_stack([10*np.cos(theta) + np.random.normal(0, 0.3, 50),
                              10*np.sin(theta) + np.random.normal(0, 0.3, 50)])

X = np.vstack([cluster_1, cluster_2])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Data shape: {X_scaled.shape}")

## Step 2: Bug 1 — Incorrect eps Parameter

In [ ]:
# BUG 1: eps is too large (or too small), not calibrated to data scale
dbscan = DBSCAN(eps=100, min_samples=5)  # BUG: eps=100 is way too large
labels = dbscan.fit_predict(X_scaled)

print(f"Number of clusters: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"Number of noise points: {sum(labels == -1)}")

**Fix explanation here:**

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What the bug is:** `eps=100` is far too large for StandardScaler-normalised data, where virtually all pairwise distances are below 3.

**Why it causes wrong behaviour:** With `eps=100`, every point lies within every other point's neighbourhood. DBSCAN assigns the entire dataset to a single cluster and reports zero noise points — it cannot distinguish any structure at all.

**How the fix resolves it:** Calibrate `eps` using the k-distance graph. Compute the k-th nearest-neighbour distance for every point (where k equals `min_samples`), sort the distances in ascending order, and read `eps` from the "elbow" — the value where distances increase sharply. On this StandardScaled data the correct value will be close to 0.3, not 100.

</details>

## Step 3: Bug 2 — Missing eps Calibration

In [ ]:
# BUG 2: eps is chosen without checking the data
# (The k-distance graph is the standard way to calibrate eps)
dbscan = DBSCAN(eps=0.5, min_samples=5)  # BUG: Chosen arbitrarily without validation
labels = dbscan.fit_predict(X_scaled)

print(f"Number of clusters: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"Number of noise points: {sum(labels == -1)}")

**Fix explanation here:**

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What the bug is:** `eps=0.5` was chosen without inspecting the data's density structure — it is an arbitrary guess.

**Why it causes wrong behaviour:** A guessed `eps` may produce acceptable results on the training dataset by chance, but there is no principled reason it will generalise to new data with different density. Slight shifts in scale or distribution will break the clustering silently.

**How the fix resolves it:** Always derive `eps` from the k-distance graph on the actual dataset. The correct workflow: `nbrs = NearestNeighbors(n_neighbors=min_samples).fit(X_scaled)`, compute distances, sort, and plot — then read `eps` from the inflection point. This is evidence-based, reproducible, and transferable to new data with similar characteristics.

</details>

## Step 4: Bug 3 — Ignoring Noise Points

In [ ]:
dbscan = DBSCAN(eps=0.3, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

# BUG 3: Analysis ignores noise points
unique_labels = set(labels)
print(f"Unique labels (including noise): {unique_labels}")

for label in sorted(unique_labels):
    cluster_points = sum(labels == label)
    # BUG: This treats noise (-1) same as real clusters
    print(f"Cluster {label}: {cluster_points} points")

**Fix explanation here:**

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What the bug is:** The loop iterates over all labels including `-1` (noise) without distinguishing it from genuine cluster labels.

**Why it causes wrong behaviour:** Printing "Cluster -1: N points" implies DBSCAN found an extra cluster when in fact these are outliers the algorithm explicitly refused to assign to any cluster. This inflates the reported cluster count by one and conflates anomaly candidates with actual groups.

**How the fix resolves it:** Before processing any label, check `if label == -1: print(f"Noise points (not a cluster): {cluster_points}"); continue`. Handle noise separately — count it, investigate it, or pass it to a downstream anomaly-detection step — but never treat it as a regular cluster.

</details>

## Step 5: Corrected Pipeline with eps Calibration

In [ ]:
# The correct approach calibrates eps using the k-distance graph:
k = 5  # min_samples parameter
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

# Sort distances to the k-th nearest neighbor
distances = np.sort(distances[:, k-1], axis=0)

# Plot the k-distance graph
plt.figure(figsize=(8, 5))
plt.plot(distances)
plt.xlabel('Data Points sorted by distance')
plt.ylabel(f'{k}-th Nearest Neighbor Distance')
plt.title('K-distance Graph (helps choose eps)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Looking for the elbow in the plot above to choose eps")


<details>
<summary>🔑 Reveal summary answers</summary>

1. **eps calibration:** Use the k-distance graph to derive eps from your data — guessing produces wrong results and does not generalise.
2. **Arbitrary eps:** Even a "reasonable-sounding" value like 0.5 is arbitrary without k-distance validation; always derive it from the actual density structure.
3. **Noise handling:** Label -1 is noise, not a cluster — always separate it in analysis and treat it as anomaly candidates requiring further investigation.

</details>

## Summary

DBSCAN is powerful for non-spherical clusters, but requires careful calibration:
- **eps** must be chosen using the k-distance graph, not guessed
- **min_samples** should relate to your data density and desired cluster size
- **Noise points** (-1) are legitimate and must be handled

**Next lesson:** Hierarchical Clustering — explore cluster structure at multiple levels.